# How to Customize This Image Generation Script

This script generates rotated and mirrored letter images using PIL. Below is a complete guide to adjusting all major parameters. The explanations follow the exact order in which the configurable elements appear in the script.

---

First, the image dimensions are controlled by:

W, H = 600, 600

W is the width in pixels, and H is the height in pixels. These values determine the size of every generated image. If you change these, you may also need to adjust the letter placement coordinates in the reference layout to keep everything properly positioned.

---

Next, the background and foreground colors are defined:

BG = (255, 255, 255, 255)  
FG = (0, 0, 0, 255)

These use RGBA format: (Red, Green, Blue, Alpha), where each value ranges from 0 to 255.  
BG controls the canvas background color.  
FG controls the text color.

Examples:
- Red text: (255, 0, 0, 255)
- Blue background: (0, 0, 255, 255)
- Transparent background: (255, 255, 255, 0)

---

Then, the rotation angles are defined:

ANGLES = [0, 45, 90, 135, 180, 225, 270, 315]

These are degrees of rotation (counterclockwise). Add or remove values to control how many rotated versions are generated. For example:
- Only cardinal rotations: [0, 90, 180, 270]
- Finer rotations: [0, 30, 60, 90, 120, 150, 180]

---

Next, the letters used are defined:

LETTERS = ["R", "G"]

You can replace these with any characters or strings you want. Examples:
- Digits: ["1", "2", "3"]
- Multiple-letter strings: ["OK", "RG"]
- Other symbols: ["#", "&"]

Keep in mind that wider strings may require adjusting font size or padding.

---

The output directory is defined as:

OUT_DIR = "imgs"

This is the folder where all images will be saved. You can change this to any relative or absolute path. For example:
- "stimuli"
- "data/images/v1"
- "/Users/yourname/Desktop/output"

The script automatically creates the folder if it does not exist.

---

Font selection happens inside load_font(). The script attempts to load a TrueType font from several candidate paths. You can modify the list of font file paths inside the candidates array to prioritize a specific .ttf file on your system. If none of the listed fonts load successfully, the script falls back to a default font.

The actual font size used is set here in main():

font = load_font(size=250)

Increase this number to make letters larger. Decrease it to make them smaller. If you change the image dimensions significantly, you may want to adjust this size accordingly.

---

Placement of elements happens in two layouts:

1) Centered layout  
The rotated letter is placed at:

(W // 2, H // 2)

This is the exact center of the image. You can change this to any (x, y) coordinate if you want the centered letter shifted elsewhere.

2) Reference layout  
Two letters are placed at fixed coordinates:

left_center = (150, 300)  
right_center = (450, 300)

These are pixel positions representing the center of each letter. If you change W or H, you may want to scale these positions accordingly. A flexible alternative would be:
- left_center = (W // 4, H // 2)
- right_center = (3 * W // 4, H // 2)

That keeps the letters proportionally spaced regardless of image size.

---

Inside make_letter_layer(), padding around the text is controlled by:

pad = 30

This adds transparent space around the letter before rotation. If rotated letters look clipped, increase pad. If you want tighter cropping, decrease pad.

---

In summary, you can control:
- Image size → W, H
- Colors → BG, FG
- Rotations → ANGLES
- Letters → LETTERS
- Save location → OUT_DIR
- Font file and size → load_font() and size argument
- Layout positions → center coordinates and left/right reference coordinates
- Rotation padding → pad

Adjust these parameters to tailor the stimulus set exactly to your experiment or visualization needs.

In [1]:
import os
from PIL import Image, ImageDraw, ImageFont

W, H = 600, 600
BG = (255, 255, 255, 255)
FG = (0, 0, 0, 255)
ANGLES = [0, 45, 90, 135, 180, 225, 270, 315]
LETTERS = ["R", "G"]
OUT_DIR = "imgs"

def load_font(size=200):
    candidates = [
        "arial.ttf",
        "Arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/Library/Fonts/Arial.ttf",
        "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for p in candidates:
        try:
            return ImageFont.truetype(p, size=size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_letter_layer(letter: str, font: ImageFont.ImageFont, mirror: bool) -> Image.Image:
    tmp = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    d = ImageDraw.Draw(tmp)

    bbox = d.textbbox((0, 0), letter, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]

    pad = 30
    layer = Image.new("RGBA", (tw + 2 * pad, th + 2 * pad), (0, 0, 0, 0))
    d2 = ImageDraw.Draw(layer)

    d2.text((pad - bbox[0], pad - bbox[1]), letter, fill=FG, font=font)

    if mirror:
        layer = layer.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

    return layer


def paste_centered(base: Image.Image, layer: Image.Image, center_xy: tuple[int, int]):
    cx, cy = center_xy
    x = int(cx - layer.width / 2)
    y = int(cy - layer.height / 2)
    base.alpha_composite(layer, (x, y))


def make_centered_image(letter: str, angle: int, mirror_letter: bool, font) -> Image.Image:
    base = Image.new("RGBA", (W, H), BG)

    layer = make_letter_layer(letter, font, mirror=mirror_letter)
    rotated = layer.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True)

    paste_centered(base, rotated, (W // 2, H // 2))
    return base


def make_reference_image(letter: str, angle: int, mirror_right: bool, font) -> Image.Image:
    base = Image.new("RGBA", (W, H), BG)

    left_center = (125, 300)
    right_center = (475, 300)

    left_layer = make_letter_layer(letter, font, mirror=False)
    paste_centered(base, left_layer, left_center)

    right_layer = make_letter_layer(letter, font, mirror=mirror_right)
    right_rot = right_layer.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True)
    paste_centered(base, right_rot, right_center)

    return base


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    font = load_font(size=250)

    for letter in LETTERS:
        for angle in ANGLES:
            img_c = make_centered_image(letter, angle, mirror_letter=False, font=font)
            img_c.save(os.path.join(OUT_DIR, f"{letter}_centered_normal_{angle:03d}.png"))

            img_r = make_reference_image(letter, angle, mirror_right=False, font=font)
            img_r.save(os.path.join(OUT_DIR, f"{letter}_reference_normal_{angle:03d}.png"))

            img_c_m = make_centered_image(letter, angle, mirror_letter=True, font=font)
            img_c_m.save(os.path.join(OUT_DIR, f"{letter}_centered_mirrored_{angle:03d}.png"))

            img_r_m = make_reference_image(letter, angle, mirror_right=True, font=font)
            img_r_m.save(os.path.join(OUT_DIR, f"{letter}_reference_mirrored_{angle:03d}.png"))

    print(f"Done. Wrote 64 images to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Done. Wrote 64 images to: imgs/
